In [0]:
import time
import requests

BASE_URL = "https://api.openf1.org/v1"
_last_call = 0.0

def get_openf1(endpoint, **params):
    """Call one OpenF1 endpoint. Waits 2s between calls and retries on errors."""
    global _last_call
    for attempt in range(1, 6):
        wait = 2 - (time.time() - _last_call)
        if wait > 0:
            time.sleep(wait)
        _last_call = time.time()

        r = requests.get(f"{BASE_URL}/{endpoint}", params=params, timeout=30)

        if r.status_code == 404:
            return []
        if r.status_code == 429 or r.status_code == 500:
            print(f"  {endpoint}: HTTP {r.status_code}, retry {attempt} in {10 * attempt}s")
            time.sleep(10 * attempt)
            continue
        r.raise_for_status()
        return r.json()
    raise RuntimeError(f"{endpoint} failed after 5 attempts")


In [0]:
sessions = get_openf1("sessions", year=2025, session_type="Race")

print(f"{len(sessions)} sessions found")
for s in sessions[:6]:
    print(s["session_key"], s["date_start"][:10], s["location"], "-", s["session_name"])

In [0]:
import json
import os

LANDING = LANDING = "/Volumes/f1/bronze/landing/openf1"

def land(endpoint, data, year, session_key):
    """Save one API response, unchanged, as a JSON file in the Volume."""
    folder = f"{LANDING}/{endpoint}/year={year}"
    os.makedirs(folder, exist_ok=True)
    path= f"{folder}/session_key={session_key}.json"
    with open(path, "w") as f:
        json.dump(data, f)
    return path

In [0]:
SESSION_ENDPOINTS = ["drivers", "session_result", "laps",
                     "stints", "pit", "weather", "race_control"]

session = sessions[0]
year = session["year"]
key = session["session_key"]

for endpoint in SESSION_ENDPOINTS:
    data = get_openf1(endpoint, session_key=key)
    path = land(endpoint, data, year, key)
    print(f"{endpoint:15} {len(data):6} rows -> {path}")

In [0]:
display(spark.read.option("multiline", True).json(f"{LANDING}/laps/"))

In [0]:
from datetime import datetime, timezone

def session_is_landed(year, session_key):
    """race_control is fetched last, so if its file exists the whole session is done."""
    return os.path.exists(f"{LANDING}/race_control/year={year}/session_key={session_key}.json")

def land_session(session):
    """Fetch and land every endpoint for one session."""
    year, key = session["year"], session["session_key"]
    for endpoint in SESSION_ENDPOINTS:
        data = get_openf1(endpoint, session_key=key)
        land(endpoint, data, year, key)


In [0]:
YEAR = 2025
now = datetime.now(timezone.utc)

sessions = get_openf1("sessions", year=YEAR, session_type="Race")

for s in sessions:
    label = f'{s["session_key"]} {s["location"]} - {s["session_name"]}'

    if datetime.fromisoformat(s["date_end"]) > now:
        print(f"skip  {label} (not finished yet)")
        continue
    if session_is_landed(s["year"], s["session_key"]):
        print(f"skip  {label} (already landed)")
        continue

    land_session(s)
    print(f"done {label}")

In [0]:
for endpoint in ["meetings", "sessions"]:
    data = get_openf1(endpoint, year=YEAR)
    folder = f"{LANDING}/{endpoint}/year={YEAR}"
    os.makedirs(folder, exist_ok=True)
    with open(f"{folder}/{endpoint}.json", "w") as f:
        json.dump(data, f)
    print(f"{endpoint}: {len(data)} rows")

In [0]:
races = get_openf1("sessions", year=2025, session_type="Race")
melbourne = races[0]

grid = get_openf1("starting_grid", meeting_key=melbourne["meeting_key"])
print(f"{len(grid)} rows")
if grid:
    print(grid[0])
    print("race session_key:", melbourne["session_key"], "| grid session_key:", grid[0]["session_key"])

In [0]:
def land_starting_grids(year):
    """The grid belongs to the weekend (it comes from qualifying), so land one file per meeting."""
    now = datetime.now(timezone.utc)
    races = get_openf1("sessions", year=year, session_type="Race")
    meeting_keys = sorted({s["meeting_key"] for s in races
                           if datetime.fromisoformat(s["date_end"]) < now})
    
    folder = f"{LANDING}/starting_grid/year={year}"
    os.makedirs(folder, exist_ok=True)

    for mk in meeting_keys:
        path = f"{folder}/meeting_key={mk}.json"
        if os.path.exists(path):
            continue
        grid = get_openf1("starting_grid", meeting_key=mk)
        if not grid:
            print(f"meeting {mk}: no grid yet, will retry next run")
            continue
        with open(path, "w") as f:
            json.dump(grid, f)
        print(f"meeting {mk}: {len(grid)} rows")

land_starting_grids(YEAR)

In [0]:
def ingest_year(year):
    now = datetime.now(timezone.utc)

    # Season-level lists (small, so just refresh them every run)
    for endpoint in ["meetings", "sessions"]:
        data = get_openf1(endpoint, year=year)
        folder = f"{LANDING}/{endpoint}/year={year}"
        os.makedirs(folder, exist_ok=True)
        with open(f"{folder}/{endpoint}.json", "w") as f:
            json.dump(data, f)
        print(f"{endpoint}: {len(data)} rows")

    # Every finished race and sprint session
    for s in get_openf1("sessions", year=year, session_type="Race"):
        label = f'{s["session_key"]} {s["location"]} - {s["session_name"]}'
        if datetime.fromisoformat(s["date_end"]) > now:
            print(f"skip  {label} (not finished yet)")
        elif session_is_landed(s["year"], s["session_key"]):
            print(f"skip  {label} (already landed)")
        else:
            land_session(s)
            print(f"done  {label}")

    # Starting grids, one file per weekend
    land_starting_grids(year)

for year in [2023, 2024, 2026]:
    print(f"\n===== {year} =====")
    ingest_year(year)